In [7]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import math as mt

In [17]:
x=np.array([1, 2, 3, 2, -9,4 ,10, -3, -4])

np.std(x)

5.120763831912406

In [9]:
df = pd.read_csv('Indices_Download_2026.csv')
df['Date'] = pd.to_datetime(df['Date'])

#df = df.set_index('Date')

#df = df.loc['2025-01-02':'2026-01-15']


start = '2025-01-02'
end = '2026-01-15'

df_filtered = df[(df['Date'] >= start) & (df['Date'] <= end)]



df_new = df_filtered[['Date','^GSPC']]

print(df_new)

          Date        ^GSPC
0   2025-01-02  5868.549805
1   2025-01-03  5942.470215
2   2025-01-06  5975.379883
3   2025-01-07  5909.029785
4   2025-01-08  5918.250000
..         ...          ...
255 2026-01-09  6966.279785
256 2026-01-12  6977.270020
257 2026-01-13  6963.740234
258 2026-01-14  6926.600098
259 2026-01-15  6944.470215

[260 rows x 2 columns]


In [11]:
df_new['returns'] = np.log(df['^GSPC'] / df['^GSPC'].shift(1))
df_new = df_new.dropna()

print(df_new)

          Date        ^GSPC   returns
1   2025-01-03  5942.470215  0.012517
2   2025-01-06  5975.379883  0.005523
3   2025-01-07  5909.029785 -0.011166
4   2025-01-08  5918.250000  0.001559
5   2025-01-10  5827.040039 -0.015532
..         ...          ...       ...
255 2026-01-09  6966.279785  0.006455
256 2026-01-12  6977.270020  0.001576
257 2026-01-13  6963.740234 -0.001941
258 2026-01-14  6926.600098 -0.005348
259 2026-01-15  6944.470215  0.002577

[259 rows x 3 columns]


C:\Users\migue\AppData\Local\Temp\ipykernel_36132\2534920352.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new['returns'] = np.log(df['^GSPC'] / df['^GSPC'].shift(1))


In [42]:
sigma_zero= df_new['returns'].std()

EWMA_lambda1=0.72

EWMA_lambda2= (1-EWMA_lambda1)

n=len(df_new['returns'])

vol_EWMA=np.zeros(n)

vol_EWMA[0] = sigma_zero
returns= df_new['returns'].to_numpy()


for i in range(0,n-1):
    vol_EWMA[(i+1)]=mt.sqrt((vol_EWMA[i]**2)*EWMA_lambda1+(returns[i]**2)*EWMA_lambda2)

#print(sigma_zero)
#print(returns[0:10])
#print(vol_EWMA[0:10])

print(vol_EWMA[n-10:n+10])

[0.00522435 0.00454476 0.00510931 0.00542966 0.00495491 0.00420458
 0.00493901 0.0042731  0.00376851 0.00426994]


In [55]:
import math as mt
factor = norm.ppf(1-0.99)
print(factor)

-2.3263478740408408


In [57]:
df_new['VaR_t']=df_new['vol_21D']*mt.sqrt(10)*factor

print(df_new)

           Date        ^GSPC   returns   vol_21D     VaR_t
2338 2025-02-03  5994.569824 -0.007638  0.008841 -0.065037
2339 2025-02-04  6037.879883  0.007199  0.008913 -0.065566
2340 2025-02-05  6061.479980  0.003901  0.008565 -0.063006
2341 2025-02-06  6083.569824  0.003638  0.008524 -0.062707
2342 2025-02-07  6025.990234 -0.009510  0.008414 -0.061899
...         ...          ...       ...       ...       ...
2573 2026-01-09  6966.279785  0.006455  0.005953 -0.043792
2574 2026-01-12  6977.270020  0.001576  0.005804 -0.042694
2575 2026-01-13  6963.740234 -0.001941  0.005819 -0.042811
2576 2026-01-14  6926.600098 -0.005348  0.005408 -0.039786
2577 2026-01-15  6944.470215  0.002577  0.005397 -0.039701

[240 rows x 5 columns]
